In [ ]:
# ============================================================
# CELL — NP ↔ NI‑DAQ DRIFT QC (1 Hz SYNC LINE)
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# ---------- PATHS ----------
ni_folder = r"G:\Kevin\2026-03-05_09-27-03\Record Node 106\experiment2\recording1\events\NI-DAQmx-105.PXIe-6341\TTL"
np_folder = r"G:\Kevin\2026-03-05_09-27-03\Record Node 101\experiment2\recording1\events\Neuropix-PXI-100.ProbeA\TTL"

# ---------- SYNC BITS ----------
bit_ni = 5   # NI‑DAQ sync bit (your wiring)
bit_np = 0   # Neuropixels sync bit (IMEC maps sync to bit 0)

# ============================================================
# LOAD TTL FOLDERS
# ============================================================

def load_ttl_folder(folder):
    ts = np.load(os.path.join(folder, "timestamps.npy"))
    states = np.load(os.path.join(folder, "states.npy"))
    full_words = np.load(os.path.join(folder, "full_words.npy"))
    return ts, states, full_words

def rising_times_for_bit(timestamps, full_words, bit):
    mask = 1 << bit
    bit_on = (full_words & mask) > 0
    bit_on_prev = np.roll(bit_on, 1)
    bit_on_prev[0] = False
    rising_idx = (~bit_on_prev) & bit_on
    return timestamps[rising_idx]

# Load NI‑DAQ TTLs
ts_ni, states_ni, words_ni = load_ttl_folder(ni_folder)
rising_ni = rising_times_for_bit(ts_ni, words_ni, bit_ni)

# Load Neuropixels TTLs
ts_np, states_np, words_np = load_ttl_folder(np_folder)
rising_np = rising_times_for_bit(ts_np, words_np, bit_np)

print(f"NI rising edges: {len(rising_ni)}")
print(f"NP rising edges: {len(rising_np)}")

# ============================================================
# ALIGN PULSES BY INDEX
# ============================================================

n = min(len(rising_ni), len(rising_np))
t_ni = rising_ni[:n]
t_np = rising_np[:n]

print(f"Using {n} matched pulses for drift fit")

# ============================================================
# LINEAR FIT: t_np = a + b * t_ni
# ============================================================

A = np.vstack([t_ni, np.ones_like(t_ni)]).T
b_fit, a_fit = np.linalg.lstsq(A, t_np, rcond=None)[0]

drift_ppm = (b_fit - 1.0) * 1e6

print("\n===== DRIFT FIT RESULTS =====")
print(f"Offset (a): {a_fit:.6f} s")
print(f"Drift factor (b): {b_fit:.9f}")
print(f"Drift: {drift_ppm:.2f} ppm")

# ============================================================
# RESIDUALS (QUALITY CHECK)
# ============================================================

t_np_pred = a_fit + b_fit * t_ni
residuals = t_np - t_np_pred

plt.figure(figsize=(12,4))
plt.plot(t_ni, residuals * 1000, '.', markersize=2)
plt.axhline(0, color='k', linewidth=1)
plt.xlabel("NI‑DAQ time (s)")
plt.ylabel("Residual (ms)")
plt.title("NP ↔ NI‑DAQ Sync Residuals (after drift correction)")
plt.tight_layout()
plt.show()

print("\n===== DRIFT QC COMPLETE =====\n")